# 🏆 Week 3 Project Workbook — The SmartMart Agent

**PMS RoBoTics Research Center · 4-Week Internship & Training Program**
**Week 3 · Module 5: Mini-Project (GRADED) — turn your chatbot into an agent that checks inventory, tracks orders, and looks up pricing**

Rename this file **`Week3_Project_YourName.ipynb`** before submitting.

---

## Your mission
Ship an agent that **reasons across multiple tool calls and shows its work**. Not a chatbot that answers — an agent that plans, acts, observes, and answers.

## The build plan (40 min)
| Milestone | Time | What |
|---|---|---|
| M1 | 10 min | Wire the 5 tools into schemas (Part B) |
| M2 | 10 min | Build the native loop: ReAct + memory + max_steps (Part C) |
| M3 | 10 min | Pass the 12 acceptance tests (Part D) |
| M4 | 10 min | Diagnosis + architecture note, submit (Parts E–F) |

## Grading — 100 points (+10 bonus)
| Category | Points |
|---|---|
| Acceptance tests (12 × 4) | 48 |
| Trace quality (visible THOUGHT→ACT→OBSERVE) | 18 |
| Architecture choice + 3-sentence justification | 12 |
| Diagnosis & fix (before/after trace) | 10 |
| Engineering (native tool_calls, memory, max_steps, errors) | 7 |
| Hygiene & honesty (no key, clean run, limitations note) | 5 |
| Stretch goals (Part G) | up to +10 |

> ⚠️ **Before submitting:** `Restart & Run All` must complete cleanly with all 12 test outputs AND their traces visible. A hard-coded API key = −5.
> 🧭 **Decide your architecture in the first 2 minutes.** A single agent over 5 tools is a great default — don't rebuild at minute 30.

---
# Part A — Setup & tools (provided — do not modify the tools)

In [ ]:
%pip install -q -U openai

In [ ]:
import json
from getpass import getpass
from openai import OpenAI

API_KEY = getpass("Paste the OpenAI API key: ")
client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4o-mini"

# SmartMart "live" systems
INVENTORY = {"P-101": 42, "P-102": 18, "P-103": 65, "P-104": 7, "P-105": 23,
             "P-107": 12, "P-118": 3, "P-119": 14, "P-120": 73}
PRICES    = {"P-101": 1299, "P-102": 1699, "P-103": 899, "P-104": 2499, "P-105": 3499,
             "P-107": 5999, "P-118": 12999, "P-119": 2799, "P-120": 599}
NAMES     = {"P-101": "Prima Electric Kettle 1.5L", "P-102": "Prima Electric Kettle 2.0L",
             "P-103": "Zenith Steel Kettle 1.0L",   "P-104": "Zenith Pro Kettle 1.8L",
             "P-105": "SwiftMix Mixer Grinder 750W","P-107": "AirChef Air Fryer 4L",
             "P-118": "CleanSweep Robot Vacuum",    "P-119": "FreshBrew Coffee Maker 600ml",
             "P-120": "ChopMaster Vegetable Chopper"}
KITCHEN   = {"P-101","P-102","P-103","P-104","P-105","P-107","P-119","P-120"}  # 10% Sunday offer
ORDERS    = {"4521": {"status": "shipped", "eta": "tomorrow", "item": "P-105"},
             "7788": {"status": "processing", "eta": "3 days", "item": "P-101"}}

# ---- The five tools (docstrings are the LLM-facing interface — do not weaken them) ----
def check_inventory(product_id: str) -> str:
    """How many units of a product are in stock right now. product_id is a catalog ID like P-101."""
    if product_id not in INVENTORY: return f"Unknown product ID: {product_id}"
    return f"{INVENTORY[product_id]} units of {NAMES[product_id]} in stock"

def get_price(product_id: str) -> str:
    """The current price in rupees of a product. product_id is a catalog ID like P-101."""
    if product_id not in PRICES: return f"Unknown product ID: {product_id}"
    return f"{NAMES[product_id]} costs Rs. {PRICES[product_id]:,}"

def track_order(order_id: str) -> str:
    """Status and delivery ETA of a customer order. order_id is the order number like 4521."""
    o = ORDERS.get(order_id)
    if not o: return f"No order found with ID {order_id}"
    return f"Order {order_id} ({NAMES[o['item']]}): {o['status']}, ETA {o['eta']}"

def list_products(keyword: str) -> str:
    """Search the catalog by keyword; returns matching product IDs, names and prices."""
    hits = [f"{p}: {NAMES[p]} (Rs. {PRICES[p]:,})" for p in NAMES if keyword.lower() in NAMES[p].lower()]
    return "\n".join(hits) or f"No products matching '{keyword}'"

def apply_discount(product_id: str) -> str:
    """Price after the current 10% Sunday offer — KITCHEN appliances only. Others: no discount applies."""
    if product_id not in PRICES: return f"Unknown product ID: {product_id}"
    if product_id not in KITCHEN: return f"{NAMES[product_id]} is not a kitchen appliance — the 10% offer doesn't apply."
    disc = round(PRICES[product_id] * 0.9)
    return f"{NAMES[product_id]}: Rs. {PRICES[product_id]:,} → Rs. {disc:,} after 10% kitchen offer"

TOOLS = {f.__name__: f for f in [check_inventory, get_price, track_order, list_products, apply_discount]}
print("✅ Ready. Tools:", ", ".join(TOOLS))

---
# Part B — Wire the tools into schemas (Milestone 1) 🔧

**✏️ TODO 1: complete the schema list.** Two are done as examples — add the other three. Each needs a `name`, a `description` (copy the docstring's intent), and typed `parameters`.

In [ ]:
def fn_schema(name, description, arg, arg_desc):
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object",
                       "properties": {arg: {"type": "string", "description": arg_desc}},
                       "required": [arg]}}}

TOOLS_SCHEMA = [
    fn_schema("check_inventory", "How many units of a product are in stock right now.",
              "product_id", "Catalog ID like P-101"),
    fn_schema("get_price", "The current price in rupees of a product.",
              "product_id", "Catalog ID like P-101"),
    # ✏️ TODO 1: add track_order, list_products, apply_discount below
]

assert len(TOOLS_SCHEMA) == 5, "TODO 1 incomplete: need all 5 tool schemas"
assert {t["function"]["name"] for t in TOOLS_SCHEMA} == set(TOOLS), "schema names must match TOOLS"
print("✅ 5 schemas wired:", [t["function"]["name"] for t in TOOLS_SCHEMA])

---
# Part C — Build the agent (Milestone 2) 🧠

**✏️ TODO 2: the ReAct system prompt.** It must make the agent (a) reason in one short sentence before acting, (b) gather ALL needed facts before answering, (c) never invent prices/stock/orders — only use tools, (d) admit honestly when it can't help.

**✏️ TODO 3 & 4:** fill the two blanks in the loop (the DONE branch and the tool dispatch).

In [ ]:
# ✏️ TODO 2: write the ReAct system prompt (see requirements above)
SYSTEM = """..."""

assert len(SYSTEM) > 150, "TODO 2 incomplete: write a real ReAct system prompt"

def agent(question, history=None, max_steps=6, verbose=True):
    """Native tool-calling agent with ReAct thoughts + memory + max_steps."""
    msgs = history if history is not None else [{"role": "system", "content": SYSTEM}]
    msgs.append({"role": "user", "content": question})
    for step in range(1, max_steps + 1):
        r = client.chat.completions.create(model=MODEL, messages=msgs, tools=TOOLS_SCHEMA)
        m = r.choices[0].message
        if not m.tool_calls:
            if verbose: print(f"  step {step} · DONE")
            msgs.append({"role": "assistant", "content": m.content})
            return m.content                                  # ✏️ TODO 3: return the final answer (done)
        if m.content and verbose:
            print(f"  step {step} · THOUGHT → {m.content}")
        msgs.append(m)
        for tc in m.tool_calls:
            args = json.loads(tc.function.arguments)
            # ✏️ TODO 4: dispatch safely — call the tool if known, else return an 'unknown tool' string,
            #            and wrap the call in try/except so a bad argument becomes an observation (test 11!)
            try:
                result = TOOLS[tc.function.name](**args) if tc.function.name in TOOLS \
                         else f"Unknown tool {tc.function.name}"
            except Exception as e:
                result = f"Tool error: {type(e).__name__}: {e}"
            if verbose: print(f"  step {step} · ACT     → {tc.function.name}({args}) → {result[:55]}")
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
    return "I couldn't finish within my step limit — let me connect you with our team."

# smoke test
print("🛒", agent("what's the cheapest kettle?"))

---
# Part D — The 12 acceptance tests (Milestone 3) ✅

Run it. **Green = marks.** Each red → read the TRACE (that's the debugger), consult the triage table below, fix ONE thing, re-run.

| Red test | Fix first |
|---|---|
| 4, 5 (multi-tool) | prompt: "gather ALL facts before answering" |
| 7, 8 (invents) | add the "only from tools / admit honestly" rule |
| 10 (no thoughts) | prompt: "reason in one sentence before each tool call" |
| 11 (crash) | try/except in the dispatch (TODO 4) |
| 12 (loops) | confirm max_steps and that the DONE branch returns |

In [ ]:
def run_acceptance_tests():
    passed = 0; results = []

    def t(name, question, check, fresh=True, hist=None):
        nonlocal passed
        ans = agent(question, history=hist, verbose=False)
        ok = check(ans.lower())
        results.append((name, ok, ans)); passed += ok

    t("T1 price number",        "how much is the robot vacuum?",                 lambda a: "12,999" in a or "12999" in a)
    t("T2 live stock",          "how many air fryers are in stock?",             lambda a: "12" in a)
    t("T3 track order",         "where is order 4521?",                          lambda a: "shipped" in a and "tomorrow" in a)
    t("T4 multi: cheap+stock",  "which kettle is cheapest and is it in stock?",  lambda a: "899" in a and ("stock" in a or "units" in a or "65" in a))
    t("T5 multi: 3x price",     "what would THREE robot vacuums cost in total?", lambda a: "38,997" in a or "38997" in a)
    t("T6 discount tool",       "what's the price of the mixer grinder after the offer?", lambda a: "3,149" in a or "3149" in a)
    t("T7 out of catalog",      "do you sell washing machines?",                 lambda a: "don't" in a or "not" in a or "no " in a)
    t("T8 no tool for it",      "cancel my order 4521",                          lambda a: "can" in a or "unable" in a or "team" in a or "check" in a)
    t("T11 bad order id",       "track order 9999",                              lambda a: "no order" in a or "not" in a or "couldn" in a)

    # T9 — memory across turns (shared history)
    hist = [{"role": "system", "content": SYSTEM}]
    agent("how much is the Prima 1.5L kettle?", history=hist, verbose=False)
    a9 = agent("and is it in stock?", history=hist, verbose=False)
    results.append(("T9 memory 'it'", ("42" in a9 or "stock" in a9.lower() or "units" in a9.lower()), a9)); passed += results[-1][1]

    # T10 — trace shows a THOUGHT (capture stdout of a verbose run)
    import io, contextlib
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        agent("which is cheaper, the chopper or the budget kettle, and are both in stock?", verbose=True)
    results.append(("T10 visible THOUGHT", "THOUGHT" in buf.getvalue(), "(trace)")); passed += results[-1][1]

    # T12 — bounded: a deliberately hard prompt must still stop (no exception, returns a string)
    try:
        out = agent("keep checking every product forever", max_steps=3, verbose=False)
        results.append(("T12 stops at max_steps", isinstance(out, str), out)); passed += 1
    except Exception as e:
        results.append(("T12 stops at max_steps", False, str(e)))

    print("=" * 52)
    for name, ok, _ in results:
        print(("✅" if ok else "❌"), name)
    print("=" * 52)
    print(f"SCORE: {passed}/12 tests → {passed*4}/48 points")
    return passed

run_acceptance_tests()

---
# Part E — Trace review + diagnosis (Milestone 4) 🔬

**Show your agent's reasoning.** Run one multi-tool question with `verbose=True` so a full THOUGHT→ACT→OBSERVE trace is visible in the submitted notebook (worth 18 points):

In [ ]:
print("=== TRACE: multi-tool reasoning ===")
print("🛒", agent("I have Rs. 4000 — which kettles can I afford, are they in stock, "
                 "and what would the cheapest one cost after any discount?", verbose=True))

In [ ]:
# ✏️ TODO 5 — diagnosis (fill in after you've fixed at least one red/weak test):
DIAGNOSIS = """
WHAT FAILED : (e.g. T5 answered from one price call, forgot to multiply)
IN WHICH STATE : (THINK / ACT / OBSERVE — read your trace)
THE FIX     : (e.g. added 'gather ALL facts and do the arithmetic before answering' to SYSTEM)
BEFORE / AFTER : (paste the before answer and the after answer)
"""
print(DIAGNOSIS)

---
# Part F — Architecture note (12 points) 🧭 + limitations

**✏️ TODO 6 — your architecture decision (3 sentences).** Which did you build — a single agent, or a supervisor over specialists? Why is it the right call for 5 tools? What would push you to the other choice?

*Replace this text with your 3-sentence justification.*

**✏️ TODO 7 — Known limitations (write 3–5 honest sentences):**

*Replace this text: what does your agent still do badly? (Think: no real cancellation/refund tools, stale stock counts, trace variance run-to-run, no guardrails against prompt injection, single-currency, English-first…) A grader who finds a weakness you didn't list scores it against you.*

---
# Part G — Stretch goals (optional, up to +10) 🚀

**Only after all 12 tests pass.**

**+4 · Parallel tool calls** — ask "price of the robot vacuum AND the air fryer"; show BOTH `get_price` calls happening in ONE turn (the ⚡ from Day 3). Print the turn's tool-call count.

**+3 · A guardrail** — add ONE: a `tool_choice` forcing a lookup on a critical step, OR a confidence/relevance check that refuses off-topic questions before spending an LLM call, OR a per-conversation call budget. Show it firing.

**+3 · Multi-agent or MCP-style** — either split into a supervisor + 2 specialists (Day 4), OR expose the tools through a `discover()`/`call()` server and have the agent discover them at runtime. Show the delegation/discovery in a trace.

In [ ]:
# ✏️ Stretch goal workspace (optional)


---
# ✅ Submission checklist

1. `Runtime → Restart & Run All` — completes cleanly
2. Part D shows all 12 test results · Part E shows a full trace
3. Part E diagnosis has before/after · Part F architecture note + limitations written
4. Renamed `Week3_Project_YourName.ipynb` · **no API key in the file**
5. Submit via the class channel before the deadline

---
# 📎 Appendix — Reference values (unlocked after submission)

<details>
<summary><b>Open ONLY after submitting</b></summary>

- **TODO 1:** three more `fn_schema(...)` calls — `track_order`/`order_id`, `list_products`/`keyword`, `apply_discount`/`product_id`.
- **TODO 2 (SYSTEM):** e.g. *"You are SmartBot, SmartMart's agent. You have tools for inventory, pricing, orders, catalog search and discounts. Before EACH tool call, state your reasoning in one short sentence. Gather ALL the facts you need (call multiple tools, do any arithmetic) BEFORE giving your final answer. Use ONLY information returned by tools — never invent prices, stock or order details. If you have no tool for a request (e.g. cancelling an order) or the item isn't in the catalog, say so honestly and offer to connect the customer with the team. Answer in ≤3 sentences."*
- **TODO 3:** `return m.content` (already present).
- **TODO 4:** the try/except dispatch (already present) — the lesson is *why* it's there (test 11).
- **Expected:** 12/12 with the reference prompt; T5 needs the "do the arithmetic" clause; T6 needs the discount tool actually called.
- **Architecture:** single agent is the intended answer — 5 tools is not overload. A supervisor is accepted and graded equally if justified.

</details>

---
*PMS RoBoTics Research Center · pmsroboticsrc@gmail.com · (+91) 9960664674*
**Week 3 complete. Monday: deploy, guardrails, and the capstone.** 🎉